# Off-Grid Solar Feasibility – Northern Nigeria

**Mohammed Tukur Saidu, PhD**  
Energy & Community Development Consultant  

This notebook reproduces the core analytical steps used in the four pilot feasibility studies.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from survey_prep import construct_indicators, gender_disaggregated_summary
from techno_economic import lcoe, npv, sensitivity_table, diesel_baseline_cost
from esg_scoring import gender_action_score, es_risk_screen

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

## 1. Load survey data & construct indicators

In [ ]:
df = pd.read_csv("../data/processed_indicators.csv")
print(df.shape)
df.head()

## 2. Gender-lens descriptive analysis

In [ ]:
gender_summary = gender_disaggregated_summary(df)
display(gender_summary)

print("\nGender Action Score:")
print(gender_action_score(df))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df, x="respondent_gender", y="wtp_solar_monthly_ngn", ax=axes[0])
axes[0].set_title("Willingness-to-Pay by Gender")
sns.boxplot(data=df, x="respondent_gender", y="energy_burden", ax=axes[1])
axes[1].set_title("Energy Burden by Gender")
plt.tight_layout()
plt.show()

## 3. Techno-economic modelling (illustrative mini-grid)

In [ ]:
# Example assumptions (replace with site-specific engineering data)
CAPEX = 85_000_000          # NGN – 50 kW system
OPEX  = 4_500_000           # annual
GEN   = 95_000              # kWh/year
DIESEL_PRICE = 850          # NGN/litre

solar_lcoe = lcoe(CAPEX, OPEX, GEN, lifetime=20, discount_rate=0.12)
diesel_cost = diesel_baseline_cost(DIESEL_PRICE)

print(f"Solar LCOE:   ₦{solar_lcoe:,.1f} / kWh")
print(f"Diesel cost:  ₦{diesel_cost:,.1f} / kWh")
print(f"Cost advantage: {diesel_cost/solar_lcoe:.1f}×")

In [ ]:
sens = sensitivity_table(
    base_capex=CAPEX,
    base_opex=OPEX,
    base_gen=GEN,
    param_ranges={
        "capex": [0.8, 0.9, 1.0, 1.1, 1.2],
        "discount_rate": [0.08, 0.10, 0.12, 0.14, 0.16],
    }
)
display(sens)

## 4. E&S risk screen (illustrative)

In [ ]:
community_profile = {
    "displaced_hh": 0,
    "contractor_workers": 15,
}
print(es_risk_screen(community_profile))

---
**Usage note**  
Replace synthetic survey data with real field data (after ethical clearance).  
The techno-economic functions accept any consistent currency units.